# 🐼 Mastering Pandas: Hands-On Practice Notebook
### *Foundations of Series, DataFrames, Slicing, and Multi-Format I/O with Real-World Scientific & Economic Data*

---

## 🎯 Pedagogical Objectives
Welcome to this hands-on **Pandas Practice Laboratory**! This notebook is designed to take you from foundational concepts to confident data manipulation using real-world data structures from three distinct empirical disciplines:
- 🌌 **Astronomy**: Stellar catalogs, celestial coordinates, parallax distances, SDSS galaxy photometry, and exoplanet properties.
- 🧬 **Biology & Bioinformatics**: RNA-Seq differential expression profiles, GC-content metrics, clinical trial cohorts, and biological pathways.
- 📈 **Economics & Finance**: Macroeconomic indicators, purchasing power decay under inflation, central bank policy rates, and currency exchange streams.

---

## 🗺️ Notebook Structure
1. [⚙️ Environment Setup & AutoGrader Initialization](#0-setup)
2. [🧩 Part 1: The Pandas `Series` (1D Labeled Array)](#1-series)
   - *Astronomy Example*: Apparent Magnitudes & Flux Calculations
   - *Exercise 1.1 (Biology)*: Gene GC-Content Analysis
   - *Exercise 1.2 (Economics)*: Inflation Decay & Purchasing Power
3. [📊 Part 2: The Pandas `DataFrame` (2D Tabular Structure)](#2-dataframe)
   - *Biology Example*: Differential Gene Expression (RNA-Seq)
   - *Exercise 2.1 (Economics)*: Global Wealth, Population & GDP per Capita
   - *Exercise 2.2 (Astronomy)*: Stellar Cluster Parallaxes & Absolute Magnitudes
4. [🔍 Part 3: Slicing & Indexing Basics](#3-slicing)
   - Understanding `.loc` vs `.iloc` vs Boolean Indexing
   - *Astronomy Example*: Exoplanet Catalog Querying
   - *Exercise 3.1 (Biology)*: Clinical Cohort Biomarkers Slicing
   - *Exercise 3.2 (Astronomy)*: Habitable Zone Exoplanet Candidates
5. [💾 Part 4: Multi-Format Input & Output (I/O) Modes](#4-io)
   - CSV, TSV, JSON (`records`, `split`), Excel (`.xlsx`), and In-Memory Streams (`io.StringIO`)
   - *Exercise 4.1 (Astronomy I/O)*: SDSS Galaxy Pipeline & TSV Export
   - *Exercise 4.2 (Biology I/O)*: Gene Expression Filtering & JSON Export
   - *Exercise 4.3 (Economics I/O)*: Parsing In-Memory Financial Stream
6. [🏆 Part 5: Capstone Multi-Domain Challenge](#5-capstone)
   - End-to-End Analysis Pipeline & Final Graduation Scorecard!

---

## 🧪 Interactive AutoEvaluation System
Every exercise is paired with an immediate **autoevaluation cell**. When you run the evaluation cell, it will:
- ✅ Confirm your logic if correct and award points.
- ❌ Provide immediate diagnostic hints if an output doesn't match expected values or if variables are missing.
- 📊 Track your cumulative score for the final report card at the end of the notebook!


In [ ]:
# ==============================================================================
# Cell 1: Environment Imports & Interactive AutoGrader Engine
# ==============================================================================
import os
import io
import json
import numpy as np
import pandas as pd

# Display settings for clean table rendering
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 1000)

class AutoGrader:
    """
    Interactive autoevaluation engine for Pandas practice exercises.
    Tracks user progress, validates data types, shapes, and numerical tolerances.
    """
    def __init__(self):
        self.results = {}
        self.total_exercises = 10

    def _record(self, ex_id, passed, message):
        self.results[ex_id] = {'passed': passed, 'message': message}
        icon = "✅ [PASS]" if passed else "❌ [FAIL]"
        print(f"{icon} {ex_id}: {message}")

    def check_1_1(self, s_gc, mean_gc, top_gc_gene):
        ex_id = "Exercise 1.1 (Biology - Gene GC Content)"
        if s_gc is None or not isinstance(s_gc, pd.Series):
            return self._record(ex_id, False, "'s_gc' is None or not a pandas Series. Did you construct it with pd.Series()?")
        if len(s_gc) != 6:
            return self._record(ex_id, False, f"'s_gc' should have 6 elements, got {len(s_gc)}.")
        if mean_gc is None or not isinstance(mean_gc, (int, float, np.number)):
            return self._record(ex_id, False, "'mean_gc' is None or not numeric. Did you compute s_gc.mean()?")
        if not np.isclose(mean_gc, 49.833333, atol=1e-2):
            return self._record(ex_id, False, f"'mean_gc' should be ~49.83, got {mean_gc}.")
        if top_gc_gene is None or str(top_gc_gene).strip().upper() != 'MYC':
            return self._record(ex_id, False, f"'top_gc_gene' should be 'MYC', got {top_gc_gene}. Hint: use s_gc.idxmax().")
        self._record(ex_id, True, "Series properly constructed, mean calculated (~49.83%), and 'MYC' correctly identified.")

    def check_1_2(self, s_inflation, s_retention, low_inflation_retention):
        ex_id = "Exercise 1.2 (Economics - Inflation Decay)"
        if s_inflation is None or not isinstance(s_inflation, pd.Series):
            return self._record(ex_id, False, "'s_inflation' is None or not a pandas Series.")
        if s_retention is None or not isinstance(s_retention, pd.Series):
            return self._record(ex_id, False, "'s_retention' is None or not a pandas Series.")
        if len(s_inflation) != 5:
            return self._record(ex_id, False, f"'s_inflation' should have 5 items, got {len(s_inflation)}.")
        expected_retention = 1.0 / (1.0 + s_inflation / 100.0)
        if not np.allclose(s_retention.values, expected_retention.values, atol=1e-3):
            return self._record(ex_id, False, "Values in 's_retention' do not match formula: 1 / (1 + s_inflation / 100).")
        if low_inflation_retention is None or not isinstance(low_inflation_retention, pd.Series):
            return self._record(ex_id, False, "'low_inflation_retention' must be a filtered pandas Series.")
        if set(low_inflation_retention.index) != {'UK', 'USA'}:
            return self._record(ex_id, False, f"Expected low-inflation countries ['UK', 'USA'], got {list(low_inflation_retention.index)}.")
        self._record(ex_id, True, "Vectorized purchasing power decay computed and low-inflation countries filtered.")

    def check_2_1(self, df_econ, df_econ_sorted):
        ex_id = "Exercise 2.1 (Economics - Wealth & Productivity)"
        if df_econ is None or not isinstance(df_econ, pd.DataFrame):
            return self._record(ex_id, False, "'df_econ' is None or not a pandas DataFrame.")
        if 'GDP_per_Capita_USD' not in df_econ.columns:
            return self._record(ex_id, False, "Missing column 'GDP_per_Capita_USD' in df_econ.")
        if 'High_Income' not in df_econ.columns:
            return self._record(ex_id, False, "Missing boolean column 'High_Income' in df_econ.")
        if df_econ_sorted is None or not isinstance(df_econ_sorted, pd.DataFrame):
            return self._record(ex_id, False, "'df_econ_sorted' is None or not a DataFrame.")
        if len(df_econ_sorted) != 5:
            return self._record(ex_id, False, f"'df_econ_sorted' should have 5 rows, got {len(df_econ_sorted)}.")
        if df_econ_sorted.iloc[0]['Country'] != 'Germany' or df_econ_sorted.iloc[-1]['Country'] != 'India':
            return self._record(ex_id, False, "df_econ_sorted is not correctly sorted descending by GDP_per_Capita_USD.")
        self._record(ex_id, True, "DataFrame built, GDP per capita computed, income threshold flagged, and sorted.")

    def check_2_2(self, df_stars, mean_hot_temp):
        ex_id = "Exercise 2.2 (Astronomy - Stellar Cluster Distance)"
        if df_stars is None or not isinstance(df_stars, pd.DataFrame):
            return self._record(ex_id, False, "'df_stars' is None or not a DataFrame.")
        if 'Distance_pc' not in df_stars.columns or 'Abs_V_mag' not in df_stars.columns:
            return self._record(ex_id, False, "df_stars must contain columns 'Distance_pc' and 'Abs_V_mag'.")
        if not np.allclose(df_stars['Distance_pc'].values, [400.0, 800.0, 200.0, 1250.0, 320.0], atol=1e-2):
            return self._record(ex_id, False, "Distance_pc values do not match 1000.0 / Parallax_mas.")
        if mean_hot_temp is None or not isinstance(mean_hot_temp, (int, float, np.number)):
            return self._record(ex_id, False, "'mean_hot_temp' is None or not numeric.")
        if not np.isclose(mean_hot_temp, 6433.333, atol=1e-1):
            return self._record(ex_id, False, f"mean_hot_temp should be ~6433.33 K, got {mean_hot_temp}.")
        self._record(ex_id, True, "Distances, absolute magnitudes, and subset mean temperature correctly calculated.")

    def check_3_1(self, slice_loc, slice_iloc, high_risk_patients):
        ex_id = "Exercise 3.1 (Biology - Clinical Slicing)"
        if not (isinstance(slice_loc, pd.DataFrame) and isinstance(slice_iloc, pd.DataFrame) and isinstance(high_risk_patients, pd.DataFrame)):
            return self._record(ex_id, False, "All three outputs ('slice_loc', 'slice_iloc', 'high_risk_patients') must be DataFrames.")
        if slice_loc.shape != (3, 2) or list(slice_loc.columns) != ['Age', 'Glucose_mg_dL']:
            return self._record(ex_id, False, f"slice_loc shape or columns incorrect. Expected (3, 2) and ['Age', 'Glucose_mg_dL'], got {slice_loc.shape} and {list(slice_loc.columns)}.")
        if slice_iloc.shape != (3, 2) or list(slice_iloc.columns) != ['Systolic_BP', 'Group']:
            return self._record(ex_id, False, f"slice_iloc shape or columns incorrect. Expected (3, 2) and ['Systolic_BP', 'Group'], got {slice_iloc.shape} and {list(slice_iloc.columns)}.")
        if set(high_risk_patients.index) != {'PT-102', 'PT-104'}:
            return self._record(ex_id, False, f"high_risk_patients should have patient indices ['PT-102', 'PT-104'], got {list(high_risk_patients.index)}.")
        self._record(ex_id, True, "Label slicing (.loc), positional slicing (.iloc), and boolean filtering all verified.")

    def check_3_2(self, df_habitable, habitable_names):
        ex_id = "Exercise 3.2 (Astronomy - Habitable Zone Exoplanets)"
        if df_habitable is None or not isinstance(df_habitable, pd.DataFrame):
            return self._record(ex_id, False, "'df_habitable' is None or not a DataFrame.")
        expected_planets = {'Kepler-452b', 'Proxima Centauri b', 'TRAPPIST-1e', 'TOI-700 d'}
        if set(df_habitable['Planet']) != expected_planets:
            return self._record(ex_id, False, f"Expected planets {expected_planets}, got {set(df_habitable['Planet'])}.")
        if habitable_names is None or set(habitable_names) != expected_planets:
            return self._record(ex_id, False, "habitable_names does not match expected planet list.")
        self._record(ex_id, True, "Habitable zone planets correctly isolated using compound boolean filters.")

    def check_4_1(self, df_galaxies, df_readback, output_path):
        ex_id = "Exercise 4.1 (Astronomy I/O - SDSS Galaxy Pipeline)"
        if df_galaxies is None or not isinstance(df_galaxies, pd.DataFrame):
            return self._record(ex_id, False, "'df_galaxies' is None or not a DataFrame.")
        if not os.path.exists(output_path):
            return self._record(ex_id, False, f"Output TSV file not found at '{output_path}'. Did you call to_csv(..., sep='\\t')?")
        if df_readback is None or not isinstance(df_readback, pd.DataFrame) or len(df_readback) != 6:
            return self._record(ex_id, False, f"'df_readback' should have 6 rows, got {len(df_readback) if isinstance(df_readback, pd.DataFrame) else 'None'}.")
        if not (all(df_readback['galaxy_type'] == 'Spiral') and all(df_readback['redshift'] > 0.06)):
            return self._record(ex_id, False, "df_readback contains galaxies that do not match filter criteria.")
        self._record(ex_id, True, "SDSS CSV loaded, filtered by type and redshift, saved to TSV, and verified upon readback.")

    def check_4_2(self, df_significant_genes, output_path):
        ex_id = "Exercise 4.2 (Biology I/O - Gene Expression JSON)"
        if df_significant_genes is None or not isinstance(df_significant_genes, pd.DataFrame):
            return self._record(ex_id, False, "'df_significant_genes' is None or not a DataFrame.")
        if len(df_significant_genes) != 6:
            return self._record(ex_id, False, f"'df_significant_genes' should have 6 genes, got {len(df_significant_genes)}.")
        if not os.path.exists(output_path):
            return self._record(ex_id, False, f"Output JSON file not found at '{output_path}'. Did you call to_json(..., orient='records')?")
        df_json_back = pd.read_json(output_path, orient='records')
        if len(df_json_back) != 6:
            return self._record(ex_id, False, f"Saved JSON file has {len(df_json_back)} records instead of 6.")
        self._record(ex_id, True, "Differentially expressed genes filtered and exported to JSON records format.")

    def check_4_3(self, df_fx, total_usd_volume):
        ex_id = "Exercise 4.3 (Economics I/O - StringIO Stream)"
        if df_fx is None or not isinstance(df_fx, pd.DataFrame):
            return self._record(ex_id, False, "'df_fx' is None or not a DataFrame.")
        if len(df_fx) != 5:
            return self._record(ex_id, False, f"'df_fx' should have 5 rows, got {len(df_fx)}.")
        if 'Volume_USD_Millions' not in df_fx.columns:
            return self._record(ex_id, False, "Column 'Volume_USD_Millions' missing in df_fx.")
        if total_usd_volume is None or not isinstance(total_usd_volume, (int, float, np.number)):
            return self._record(ex_id, False, "'total_usd_volume' is None or not numeric.")
        if not np.isclose(total_usd_volume, 1227.42886, atol=1e-2):
            return self._record(ex_id, False, f"total_usd_volume should be ~1227.43, got {total_usd_volume}.")
        self._record(ex_id, True, "In-memory string stream parsed, comments skipped, and trading volume calculated.")

    def check_5(self, df_capstone, output_path):
        ex_id = "Exercise 5 (Capstone - Multi-Domain Executive Summary)"
        if df_capstone is None or not isinstance(df_capstone, pd.DataFrame):
            return self._record(ex_id, False, "'df_capstone' is None or not a DataFrame.")
        expected_cols = ['country', 'continent', 'GDP_per_Capita_USD', 'Economic_Vulnerability_Score']
        if list(df_capstone.columns) != expected_cols:
            return self._record(ex_id, False, f"Expected columns {expected_cols}, got {list(df_capstone.columns)}.")
        if len(df_capstone) != 4:
            return self._record(ex_id, False, f"Expected 4 stable countries, got {len(df_capstone)}.")
        if df_capstone.iloc[0]['country'] != 'Germany':
            return self._record(ex_id, False, f"First country should be Germany, got {df_capstone.iloc[0]['country']}.")
        if not os.path.exists(output_path):
            return self._record(ex_id, False, f"Capstone JSON file not found at '{output_path}'.")
        self._record(ex_id, True, "Full capstone workflow validated: CSV read, metric derivation, multi-filtering, sorting, and JSON export.")

    def summary(self):
        passed = sum(1 for r in self.results.values() if r['passed'])
        total = self.total_exercises
        pct = (passed / total) * 100
        print("\n" + "="*65)
        print("       📊 PANDAS PRACTICE NOTEBOOK FINAL SCORECARD")
        print("="*65)
        for ex_id, r in self.results.items():
            icon = "✅ PASS" if r['passed'] else "❌ FAIL"
            print(f" [{icon}] {ex_id}")
        
        attempted_count = len(self.results)
        if attempted_count < total:
            print(f" ⚠️  Note: {total - attempted_count} exercises have not been evaluated yet.")
        
        print("-"*65)
        print(f" TOTAL COMPLETED SCORE: {passed} / {total} ({pct:.1f}%)")
        if passed == total:
            print(" 🏆 OUTSTANDING WORK! You have completely mastered Pandas core basics!")
        else:
            print(" 💡 Review the diagnostic feedback above for any failing or incomplete tasks.")
        print("="*65 + "\n")

# Instantiate global grader
grader = AutoGrader()
print("🚀 Environment loaded: pandas " + pd.__version__ + ", numpy " + np.__version__)
print("🎯 AutoGrader initialized! You are ready to begin.")


<a id="1-series"></a>
# 🧩 Part 1: The Pandas `Series` (1D Labeled Array)

---

### 📖 What is a Series?
A `pandas.Series` is a **one-dimensional labeled array** capable of holding any data type (integers, floats, strings, booleans, objects). 

Think of a Series as having three fundamental components:
```text
  Index (Labels)        Values (NumPy Array)
  ┌──────────────┐     ┌──────────────────────┐
  │   'Sirius'   │ ──► │        -1.46         │  dtype: float64
  │   'Canopus'  │ ──► │        -0.74         │  name:  'Apparent_Magnitude'
  │   'Vega'     │ ──► │         0.03         │
  └──────────────┘     └──────────────────────┘
```
1. **`values`**: An underlying 1D contiguous NumPy array.
2. **`index`**: An array of axis labels (integers, strings, dates, etc.).
3. **`name`** & **`dtype`**: Metadata describing the variable and its memory representation.

Unlike native Python lists or standard 1D NumPy arrays where items are indexed strictly by integer position ($0, 1, 2, \dots$), Pandas Series permit **semantic labeling**. When you perform arithmetic operations between two Series, Pandas automatically **aligns them by index label**, not by position!

---

### 🌌 Worked Example: Apparent Stellar Magnitudes
In observational astronomy, the brightness of stars seen from Earth is quantified by **apparent magnitude** ($m$). Note that the magnitude scale is **logarithmic and inverted**: smaller or more negative numbers correspond to brighter stars!

Let's build a Series of apparent magnitudes for the brightest night-sky stars:


In [ ]:
# --- Worked Example: Apparent Magnitudes of Brightest Stars ---
star_names = ['Sirius', 'Canopus', 'Alpha Centauri', 'Arcturus', 'Vega', 'Capella', 'Rigel', 'Betelgeuse']
magnitudes = [-1.46, -0.74, -0.27, -0.05, 0.03, 0.08, 0.13, 0.50]

# 1. Constructing the Series
s_stars = pd.Series(data=magnitudes, index=star_names, name='Apparent_Magnitude')
print("--- Stellar Magnitude Series ---")
print(s_stars)

# 2. Inspecting fundamental attributes
print(f"\nIndex: {s_stars.index}")
print(f"Values array: {s_stars.values}")
print(f"Data type: {s_stars.dtype}, Name: {s_stars.name}")

# 3. Label-based vs Positional Access
print(f"\nBrightness of Vega (by label): {s_stars['Vega']}")
print(f"Brightness of first star (by position): {s_stars.iloc[0]}")

# 4. Summary Statistics
print(f"\nMean magnitude: {s_stars.mean():.2f}")
print(f"Brightest star: {s_stars.idxmin()} (magnitude {s_stars.min()})")
print(f"Faintest in list: {s_stars.idxmax()} (magnitude {s_stars.max()})")

# 5. Vectorized Physics Operation:
# Pogson's Relation: Flux ratio relative to Vega (where m_Vega = 0.03):
# F / F_vega = 10^(-0.4 * (m - m_vega))
flux_ratio_to_vega = 10 ** (-0.4 * (s_stars - 0.03))
print("\n--- Flux Ratio relative to Vega (Vega = 1.0) ---")
print(flux_ratio_to_vega.round(2))


---
### 🧬 Exercise 1.1: Gene GC-Content Analysis (Biology)

#### Biological Context:
The **GC-content** (percentage of nitrogenous bases that are either Guanine ($G$) or Cytosine ($C$)) is a critical thermodynamic and structural property of genomic DNA. Genomic regions with high GC-content have higher melting temperatures ($T_m$), denser gene coding regions, and distinct evolutionary pressures.

#### Your Task:
Given two Python lists representing 6 key human genes and their respective GC-content percentages:
1. Create a `pd.Series` named `s_gc` where:
   - The data values are `gc_percentages`.
   - The index labels are `genes`.
   - The name of the Series is `'GC_Content'`.
2. Compute the mean GC-content across all 6 genes and store it in variable `mean_gc` (float).
3. Using the `.idxmax()` method, identify the gene with the highest GC-content and store its symbol string in `top_gc_gene`.


In [ ]:
# --- Exercise 1.1 Work Cell ---
genes = ['TP53', 'EGFR', 'BRCA1', 'MYC', 'APOE', 'TNF']
gc_percentages = [42.1, 53.8, 41.5, 58.2, 55.4, 48.0]

# ==============================================================================
# TODO: Write your code below
# ==============================================================================
s_gc = None          # 1. Create the Series
mean_gc = None       # 2. Compute the mean GC-content (s_gc.mean())
top_gc_gene = None   # 3. Identify the gene symbol with highest GC-content (s_gc.idxmax())
# ==============================================================================

# Inspect your variables:
print("s_gc:\n", s_gc)
print("mean_gc:", mean_gc)
print("top_gc_gene:", top_gc_gene)


In [ ]:
# --- AutoEvaluation Cell: Exercise 1.1 ---
try:
    grader.check_1_1(s_gc, mean_gc, top_gc_gene)
except NameError as e:
    print(f"❌ [NAME ERROR]: {e}")
    print("👉 Ensure you defined: 's_gc', 'mean_gc', and 'top_gc_gene'.")


---
### 📈 Exercise 1.2: Annual Inflation & Purchasing Power Decay (Economics)

#### Economic Context:
Inflation reduces the purchasing power of money over time. If a country experiences an annual inflation rate of $r\%$, a consumer basket that cost $1.00$ currency unit at the start of the year will require $(1 + r/100)$ at year's end. 

The **purchasing power retention factor** after 1 year represents the real value retained:
$$\text{Retention Factor} = \frac{1}{1 + \frac{r}{100}}$$

For example, at $100\%$ inflation, $\text{Retention} = \frac{1}{1 + 1.0} = 0.50$ (your money lost half its real purchasing power).

#### Your Task:
Given a dictionary of annual inflation rates for 5 economies:
```python
inflation_dict = {
    'Argentina': 211.4,
    'Turkey': 64.8,
    'Egypt': 33.7,
    'UK': 4.0,
    'USA': 3.4
}
```
1. Construct a `pd.Series` named `s_inflation` directly from `inflation_dict`.
2. Compute the vectorized purchasing power retention factor Series using the formula above:
   $$\text{s\_retention} = \frac{1}{1 + \text{s\_inflation} / 100}$$
3. Using boolean indexing on `s_retention` (or `s_inflation`), extract only the countries whose inflation rate was strictly less than $10.0\%$. Store this filtered Series in `low_inflation_retention`.


In [ ]:
# --- Exercise 1.2 Work Cell ---
inflation_dict = {
    'Argentina': 211.4,
    'Turkey': 64.8,
    'Egypt': 33.7,
    'UK': 4.0,
    'USA': 3.4
}

# ==============================================================================
# TODO: Write your code below
# ==============================================================================
s_inflation = None             # 1. Create Series from inflation_dict
s_retention = None             # 2. Compute retention factor Series
low_inflation_retention = None # 3. Filter countries with inflation < 10.0%
# ==============================================================================

# Inspect your results:
print("s_inflation:\n", s_inflation)
print("\ns_retention:\n", s_retention)
print("\nlow_inflation_retention:\n", low_inflation_retention)


In [ ]:
# --- AutoEvaluation Cell: Exercise 1.2 ---
try:
    grader.check_1_2(s_inflation, s_retention, low_inflation_retention)
except NameError as e:
    print(f"❌ [NAME ERROR]: {e}")
    print("👉 Ensure you defined: 's_inflation', 's_retention', and 'low_inflation_retention'.")


<a id="2-dataframe"></a>
# 📊 Part 2: The Pandas `DataFrame` (2D Tabular Structure)

---

### 📖 What is a DataFrame?
A `pandas.DataFrame` is a **two-dimensional labeled tabular data structure** with columns of potentially different data types. You can think of it as a spreadsheet, an SQL table, or **a dictionary of Series sharing the same index**.

```text
               Column Index (df.columns)
            ┌──────────────┬──────────────────┬─────────────────┐
            │ 'gene_symbol'│ 'log2_fold_chg'  │   'p_value'     │
  Row Index ├──────────────┼──────────────────┼─────────────────┤
  0         │    'TP53'    │      -2.15       │    1.2e-06      │
  1         │    'EGFR'    │       3.45       │    2.1e-12      │
  2         │    'MYC'     │       2.78       │    8.4e-09      │
            └──────────────┴──────────────────┴─────────────────┘
```

### 🛠️ Essential DataFrame Diagnostic Methods:
| Method / Attribute | Purpose |
| :--- | :--- |
| `df.head(n)` / `df.tail(n)` | View first or last $n$ rows (default 5). |
| `df.shape` | Tuple of `(n_rows, n_columns)`. |
| `df.columns` | Index of column names. |
| `df.dtypes` | Data types of each column (`int64`, `float64`, `object`, etc.). |
| `df.info()` | Summary of non-null counts, column data types, and memory usage. |
| `df.describe()` | Descriptive statistics (mean, std, min, quartiles, max) for numeric columns. |
| `df.sort_values(by=..., ascending=...)` | Order rows by one or more columns. |
| `df.reset_index(drop=True)` | Re-index rows sequentially from $0$ to $N-1$. |

---

### 🧬 Worked Example: RNA-Seq Differential Gene Expression
In genomics, RNA-Sequencing measures cellular messenger RNA to determine which genes are active. In a cancer drug experiment, we compare treated cells versus control cells.

Let's inspect a worked biological DataFrame:


In [ ]:
# --- Worked Example: Differential Gene Expression Analysis ---
rna_data = {
    'Symbol': ['TP53', 'BRCA1', 'EGFR', 'MYC', 'VEGFA', 'GAPDH'],
    'Base_Mean': [1420.5, 830.2, 3240.8, 5120.0, 2450.1, 15400.0],
    'Log2_Fold_Change': [-2.15, -1.82, 3.45, 2.78, 1.95, 0.05],
    'P_Value': [1.2e-6, 4.5e-5, 2.1e-12, 8.4e-9, 3.3e-4, 0.72]
}

# 1. Build DataFrame
df_genes_demo = pd.DataFrame(rna_data)
print("--- Initial DataFrame ---")
print(df_genes_demo)

# 2. Add derived column: Statistical significance score -log10(p-value)
df_genes_demo['-Log10_PVal'] = -np.log10(df_genes_demo['P_Value'])

# 3. Add categorical classification based on threshold
# Genes with Log2_Fold_Change > 1.5 are Upregulated, < -1.5 are Downregulated, else Unchanged
df_genes_demo['Regulation'] = 'Unchanged'
df_genes_demo.loc[df_genes_demo['Log2_Fold_Change'] > 1.5, 'Regulation'] = 'Upregulated'
df_genes_demo.loc[df_genes_demo['Log2_Fold_Change'] < -1.5, 'Regulation'] = 'Downregulated'

# 4. Sort by significance (-Log10_PVal) descending
df_genes_sorted = df_genes_demo.sort_values(by='-Log10_PVal', ascending=False).reset_index(drop=True)

print("\n--- Processed & Ranked Genomic DataFrame ---")
print(df_genes_sorted[['Symbol', 'Log2_Fold_Change', 'P_Value', '-Log10_PVal', 'Regulation']])


---
### 📈 Exercise 2.1: Global Wealth, Population & GDP per Capita (Economics)

#### Economic Context:
Gross Domestic Product (GDP) measures the total economic output of an economy. However, raw GDP favors populous nations. **GDP per Capita** scales output by population size, providing a standard measure of productivity and living standards:
$$\text{GDP per Capita (USD)} = \frac{\text{GDP (in USD)}}{\text{Population}} = \frac{\text{GDP\_Billion} \times 10^9}{\text{Population\_Million} \times 10^6}$$

#### Your Task:
Given the dictionary below:
```python
econ_dict = {
    'Country': ['Germany', 'Japan', 'India', 'Brazil', 'Chile'],
    'GDP_Billion_USD': [4430.0, 4210.0, 3730.0, 2170.0, 340.0],
    'Population_Million': [84.4, 124.5, 1428.0, 215.3, 19.6]
}
```
1. Create a `pd.DataFrame` named `df_econ` from `econ_dict`.
2. Add a new column `'GDP_per_Capita_USD'` to `df_econ` using the formula above.
3. Add a boolean column `'High_Income'` to `df_econ` that evaluates to `True` if `'GDP_per_Capita_USD'` is strictly greater than `30000`, and `False` otherwise.
4. Create a new DataFrame `df_econ_sorted` by sorting `df_econ` by `'GDP_per_Capita_USD'` in **descending order** (`ascending=False`), and reset the row index (`.reset_index(drop=True)`).


In [ ]:
# --- Exercise 2.1 Work Cell ---
econ_dict = {
    'Country': ['Germany', 'Japan', 'India', 'Brazil', 'Chile'],
    'GDP_Billion_USD': [4430.0, 4210.0, 3730.0, 2170.0, 340.0],
    'Population_Million': [84.4, 124.5, 1428.0, 215.3, 19.6]
}

# ==============================================================================
# TODO: Write your code below
# ==============================================================================
df_econ = None          # 1. Build DataFrame, add 'GDP_per_Capita_USD' & 'High_Income'
df_econ_sorted = None   # 4. Sort descending by GDP_per_Capita_USD and reset_index
# ==============================================================================

# Inspect results:
if df_econ is not None:
    print("df_econ:\n", df_econ)
if df_econ_sorted is not None:
    print("\ndf_econ_sorted:\n", df_econ_sorted)


In [ ]:
# --- AutoEvaluation Cell: Exercise 2.1 ---
try:
    grader.check_2_1(df_econ, df_econ_sorted)
except NameError as e:
    print(f"❌ [NAME ERROR]: {e}")
    print("👉 Ensure you defined: 'df_econ' and 'df_econ_sorted'.")


---
### 🌌 Exercise 2.2: Stellar Cluster Parallaxes & Absolute Magnitudes (Astronomy)

#### Astronomical Context:
Space astrometry missions like ESA's *Gaia* measure trigonometric **parallax** ($\varpi$) in milliarcseconds (mas). The distance $d$ in parsecs (pc) is simply:
$$d = \frac{1000}{\varpi\text{ (in mas)}}$$

The **apparent magnitude** ($m$ or $V$) is how bright a star appears from Earth, which depends on distance. The **absolute magnitude** ($M_V$) is its intrinsic luminosity — the apparent brightness it would have if placed at a standard distance of exactly 10 parsecs:
$$M_V = V - 5 \log_{10}(d) + 5$$

#### Your Task:
Given the dictionary of 5 cluster stars below:
```python
stars_dict = {
    'Star_ID': ['Cl-01', 'Cl-02', 'Cl-03', 'Cl-04', 'Cl-05'],
    'V_mag': [11.2, 12.8, 9.5, 14.1, 10.4],
    'Parallax_mas': [2.5, 1.25, 5.0, 0.8, 3.125],
    'Temp_K': [6200, 4800, 7500, 3900, 5600]
}
```
1. Create a DataFrame `df_stars` from `stars_dict`.
2. Add a new column `'Distance_pc'` computed as `1000.0 / df_stars['Parallax_mas']`.
3. Add a new column `'Abs_V_mag'` computed using the distance modulus formula above:
   `df_stars['V_mag'] - 5 * np.log10(df_stars['Distance_pc']) + 5`.
4. Calculate the average temperature (in Kelvin) of the hotter stars having `Temp_K >= 5500`. Store this scalar float in `mean_hot_temp`.


In [ ]:
# --- Exercise 2.2 Work Cell ---
stars_dict = {
    'Star_ID': ['Cl-01', 'Cl-02', 'Cl-03', 'Cl-04', 'Cl-05'],
    'V_mag': [11.2, 12.8, 9.5, 14.1, 10.4],
    'Parallax_mas': [2.5, 1.25, 5.0, 0.8, 3.125],
    'Temp_K': [6200, 4800, 7500, 3900, 5600]
}

# ==============================================================================
# TODO: Write your code below
# ==============================================================================
df_stars = None        # 1. Build DataFrame, add 'Distance_pc' and 'Abs_V_mag'
mean_hot_temp = None   # 4. Compute mean Temp_K for stars with Temp_K >= 5500
# ==============================================================================

# Inspect results:
if df_stars is not None:
    print("df_stars:\n", df_stars)
print("mean_hot_temp:", mean_hot_temp)


In [ ]:
# --- AutoEvaluation Cell: Exercise 2.2 ---
try:
    grader.check_2_2(df_stars, mean_hot_temp)
except NameError as e:
    print(f"❌ [NAME ERROR]: {e}")
    print("👉 Ensure you defined: 'df_stars' and 'mean_hot_temp'.")


<a id="3-slicing"></a>
# 🔍 Part 3: Slicing & Indexing Basics

---

### 📖 The Core Slicing Methods in Pandas
Accessing subsets of data is the most frequent operation in data analysis. Pandas provides three primary ways to slice:

| Syntax | Mechanism | Boundary Rule | Example |
| :--- | :--- | :--- | :--- |
| `df['col']` | Single column access | Returns `pd.Series` | `df['Mass']` |
| `df[['col1', 'col2']]` | Multi-column access | Returns `pd.DataFrame` | `df[['Mass', 'Radius']]` |
| `df.loc[row_label, col_label]` | **Label-based** indexing | **Both endpoints INCLUDED** | `df.loc['StarA':'StarC', 'V_mag']` |
| `df.iloc[row_pos, col_pos]` | **Position-based** integer indexing | Standard Python (**stop EXCLUDED**) | `df.iloc[0:3, 1:4]` |
| `df[condition]` | **Boolean mask** filtering | Rows where `condition == True` | `df[df['Temp'] > 5000]` |

> ⚠️ **CRITICAL PITFALL**:
> In Python boolean expressions, you write `and`, `or`, `not`. 
> In **Pandas**, you **MUST** use bitwise operators:
> - `&` for element-wise **AND**
> - `|` for element-wise **OR**
> - `~` for element-wise **NOT**
> 
> Furthermore, each sub-condition **MUST be wrapped in parentheses**:
> ```python
> # ❌ WRONG (will raise TypeError or ValueError):
> df[df['Age'] > 40 and df['Glucose'] > 100]
> 
> # ✅ CORRECT:
> df[(df['Age'] > 40) & (df['Glucose'] > 100)]
> ```

---

### 🌌 Worked Example: Exoplanet Catalog Exploration
Let's inspect how `.loc`, `.iloc`, and boolean filters operate on real exoplanetary data:


In [ ]:
# --- Worked Example: Exoplanet Slicing ---
exo_data = {
    'Planet': ['Kepler-452b', 'Proxima Centauri b', 'WASP-12b', 'TRAPPIST-1e', 'HD 209458 b'],
    'Mass_Earth': [5.0, 1.17, 450.0, 0.69, 219.0],
    'Radius_Earth': [1.63, 1.03, 20.1, 0.92, 15.4],
    'Period_days': [384.8, 11.2, 1.09, 6.1, 3.52],
    'Discovery_Method': ['Transit', 'Radial Velocity', 'Transit', 'Transit', 'Transit']
}
df_exo = pd.DataFrame(exo_data)

# Set 'Planet' as the index for label-based demonstration
df_exo_indexed = df_exo.set_index('Planet')
print("--- Indexed Exoplanet DataFrame ---")
print(df_exo_indexed)

# 1. Label-based slicing with .loc:
# Notice: 'Proxima Centauri b' to 'TRAPPIST-1e' includes BOTH endpoints!
print("\n--- .loc Slice (Proxima Centauri b to TRAPPIST-1e, columns Mass to Radius) ---")
print(df_exo_indexed.loc['Proxima Centauri b':'TRAPPIST-1e', ['Mass_Earth', 'Radius_Earth']])

# 2. Positional slicing with .iloc:
# Notice: 0:2 gets row 0 and row 1 (stop is EXCLUDED)
print("\n--- .iloc Slice (Rows 0:2, Columns 0:3) ---")
print(df_exo_indexed.iloc[0:2, 0:3])

# 3. Compound Boolean Filtering:
# Find small terrestrial planets (Radius <= 2.0) with orbital period > 10 days
mask = (df_exo_indexed['Radius_Earth'] <= 2.0) & (df_exo_indexed['Period_days'] > 10.0)
print("\n--- Boolean Filter: Small planets with Period > 10 days ---")
print(df_exo_indexed[mask])


---
### 🧬 Exercise 3.1: Clinical Biomarkers Cohort Slicing (Biology)

#### Biological Context:
In clinical metabolic trials, patients are monitored for **Fasting Glucose** (normal: $< 100\,\text{mg/dL}$, pre-diabetic: $100-125\,\text{mg/dL}$, diabetic: $> 125\,\text{mg/dL}$) and **Systolic Blood Pressure** (hypertension threshold: $\ge 140\,\text{mmHg}$).

#### Your Task:
Given the clinical trial DataFrame `df_clinic` indexed by `Patient_ID`:
```python
clinic_data = {
    'Patient_ID': ['PT-101', 'PT-102', 'PT-103', 'PT-104', 'PT-105'],
    'Age': [45, 62, 38, 59, 71],
    'Glucose_mg_dL': [95.0, 142.0, 105.0, 168.0, 118.0],
    'Systolic_BP': [120, 145, 115, 160, 135],
    'Group': ['Control', 'Treatment', 'Control', 'Treatment', 'Treatment']
}
df_clinic = pd.DataFrame(clinic_data).set_index('Patient_ID')
```

1. **Label Slice (`slice_loc`)**: Using `.loc`, extract rows from `'PT-102'` to `'PT-104'` (inclusive) and columns `['Age', 'Glucose_mg_dL']`.
2. **Positional Slice (`slice_iloc`)**: Using `.iloc`, extract the first 3 rows (indices 0, 1, 2) and the last 2 columns (`'Systolic_BP'` and `'Group'`).
3. **Boolean Query (`high_risk_patients`)**: Filter patients who present **both** elevated glucose (`Glucose_mg_dL > 110`) **AND** elevated systolic blood pressure (`Systolic_BP >= 140`). Keep all columns.


In [ ]:
# --- Exercise 3.1 Work Cell ---
clinic_data = {
    'Patient_ID': ['PT-101', 'PT-102', 'PT-103', 'PT-104', 'PT-105'],
    'Age': [45, 62, 38, 59, 71],
    'Glucose_mg_dL': [95.0, 142.0, 105.0, 168.0, 118.0],
    'Systolic_BP': [120, 145, 115, 160, 135],
    'Group': ['Control', 'Treatment', 'Control', 'Treatment', 'Treatment']
}
df_clinic = pd.DataFrame(clinic_data).set_index('Patient_ID')

# ==============================================================================
# TODO: Write your code below
# ==============================================================================
slice_loc = None           # 1. Use .loc['PT-102':'PT-104', ['Age', 'Glucose_mg_dL']]
slice_iloc = None          # 2. Use .iloc[:3, [2, 3]] (or .iloc[0:3, -2:])
high_risk_patients = None  # 3. Boolean filter: Glucose > 110 AND Systolic_BP >= 140
# ==============================================================================

# Inspect results:
print("slice_loc:\n", slice_loc)
print("\nslice_iloc:\n", slice_iloc)
print("\nhigh_risk_patients:\n", high_risk_patients)


In [ ]:
# --- AutoEvaluation Cell: Exercise 3.1 ---
try:
    grader.check_3_1(slice_loc, slice_iloc, high_risk_patients)
except NameError as e:
    print(f"❌ [NAME ERROR]: {e}")
    print("👉 Ensure you defined: 'slice_loc', 'slice_iloc', and 'high_risk_patients'.")


---
### 🌌 Exercise 3.2: Filtering Habitable Zone Exoplanet Candidates (Astronomy)

#### Astronomical Context:
The **circumstellar habitable zone** (or "Goldilocks Zone") is the region around a star where liquid water could persist on an exoplanet's surface. Planetary astrophysicists look for candidates with:
1. **Rocky/Terrestrial radius**: $R_p \le 2.0\, R_\oplus$ (Earth radii).
2. **Temperate equilibrium temperature**: $T_{eq}$ between $200\,\text{K}$ and $300\,\text{K}$ ($-73^\circ\text{C}$ to $+27^\circ\text{C}$).

#### Your Task:
Given the exoplanetary catalog `df_planets`:
```python
planets_data = {
    'Planet': ['Kepler-452b', 'Proxima Centauri b', 'WASP-12b', 'TRAPPIST-1e', 'HD 209458 b', 'K2-18b', 'TOI-700 d'],
    'Mass_Earth': [5.0, 1.17, 450.0, 0.69, 219.0, 8.6, 1.25],
    'Radius_Earth': [1.63, 1.03, 20.1, 0.92, 15.4, 2.61, 1.14],
    'Period_days': [384.8, 11.2, 1.09, 6.1, 3.52, 32.9, 37.4],
    'Equilibrium_Temp_K': [265, 234, 2500, 251, 1450, 260, 269],
    'Type': ['Super-Earth', 'Terrestrial', 'Hot Jupiter', 'Terrestrial', 'Hot Jupiter', 'Sub-Neptune', 'Terrestrial']
}
df_planets = pd.DataFrame(planets_data)
```

1. Filter `df_planets` using compound boolean indexing to isolate all planets where:
   - `Radius_Earth <= 2.0`
   - **AND** `Equilibrium_Temp_K` is between $200$ and $300$ inclusive (Tip: you can use `.between(200, 300)`).
   Store the resulting filtered DataFrame in `df_habitable`.
2. Extract the `'Planet'` column from `df_habitable` as a Python list (or Series) and store it in `habitable_names`.


In [ ]:
# --- Exercise 3.2 Work Cell ---
planets_data = {
    'Planet': ['Kepler-452b', 'Proxima Centauri b', 'WASP-12b', 'TRAPPIST-1e', 'HD 209458 b', 'K2-18b', 'TOI-700 d'],
    'Mass_Earth': [5.0, 1.17, 450.0, 0.69, 219.0, 8.6, 1.25],
    'Radius_Earth': [1.63, 1.03, 20.1, 0.92, 15.4, 2.61, 1.14],
    'Period_days': [384.8, 11.2, 1.09, 6.1, 3.52, 32.9, 37.4],
    'Equilibrium_Temp_K': [265, 234, 2500, 251, 1450, 260, 269],
    'Type': ['Super-Earth', 'Terrestrial', 'Hot Jupiter', 'Terrestrial', 'Hot Jupiter', 'Sub-Neptune', 'Terrestrial']
}
df_planets = pd.DataFrame(planets_data)

# ==============================================================================
# TODO: Write your code below
# ==============================================================================
df_habitable = None    # 1. Filter planets: Radius <= 2.0 AND Temp between 200 and 300
habitable_names = None # 2. Extract list of planet names: list(df_habitable['Planet'])
# ==============================================================================

# Inspect results:
print("df_habitable:\n", df_habitable)
print("\nhabitable_names:", habitable_names)


In [ ]:
# --- AutoEvaluation Cell: Exercise 3.2 ---
try:
    grader.check_3_2(df_habitable, habitable_names)
except NameError as e:
    print(f"❌ [NAME ERROR]: {e}")
    print("👉 Ensure you defined: 'df_habitable' and 'habitable_names'.")


<a id="4-io"></a>
# 💾 Part 4: Multi-Format Input & Output (I/O) Modes

---

### 📖 The Pandas I/O Ecosystem
In real-world data science, you rarely enter data manually. You interact with disk files, web APIs, scientific archives, and databases. Pandas provides optimized, high-performance readers and writers for virtually every data format:

```text
       ┌───────────────┐
       │  CSV / TSV    │ ◄──► pd.read_csv()   / df.to_csv()
       ├───────────────┤
       │  JSON         │ ◄──► pd.read_json()  / df.to_json()
INPUT  ├───────────────┤                                      OUTPUT
──────►│  Excel (.xlsx)│ ◄──► pd.read_excel() / df.to_excel() ──────►
       ├───────────────┤
       │  In-Memory    │ ◄──► io.StringIO()   / io.BytesIO()
       ├───────────────┤
       │  Parquet      │ ◄──► pd.read_parquet()/ df.to_parquet()
       └───────────────┘
```

### 📊 Comparison of Common I/O Formats:
| Format | Reader / Writer | Compression / Speed | Schema Preserved? | Typical Use Case |
| :--- | :--- | :--- | :--- | :--- |
| **CSV** | `read_csv` / `to_csv` | Moderate / Plain text | ❌ No (types inferred) | General data exchange, astronomy catalogs |
| **TSV** | `read_csv(sep='\t')` | Moderate / Plain text | ❌ No (types inferred) | Bioinformatics (BED, GFF), astronomical tables |
| **JSON** | `read_json` / `to_json` | Flexible / Text-based | ⚠️ Partial | REST APIs, nested metadata, web apps |
| **Excel** | `read_excel` / `to_excel`| Slower / Binary zip | ⚠️ Partial | Finance, corporate reports, clinical sheets |
| **StringIO** | `io.StringIO` buffer | In-memory stream | ❌ No | Unit testing, parsing API payloads directly |

---

### 🛠️ Hands-On Demonstrations of I/O Modes
Let's inspect how to read each format from the local `data/` directory:


In [ ]:
# --- Demonstration 4A: Astronomy CSV Loading ---
# Reading a stellar catalog with custom columns and checking info
df_demo_stars = pd.read_csv('data/stellar_catalog.csv')
print("--- Loaded Stellar Catalog (CSV) ---")
print(df_demo_stars.head(3))
print("Shape:", df_demo_stars.shape)

# --- Demonstration 4B: Biology JSON Loading ---
# Reading JSON records from an RNA-Seq experiment
df_demo_genes = pd.read_json('data/gene_expression.json', orient='records')
print("\n--- Loaded Gene Expression (JSON Records) ---")
print(df_demo_genes[['symbol', 'base_mean', 'log2_fold_change', 'pathway']].head(3))

# --- Demonstration 4C: Economics Excel Loading ---
# Reading an Excel sheet of Central Bank policy interest rates
df_demo_rates = pd.read_excel('data/central_bank_rates.xlsx')
print("\n--- Loaded Central Bank Policy Rates (Excel .xlsx) ---")
print(df_demo_rates.head(3))


---
### 🌌 Exercise 4.1: SDSS Galaxy Photometry Pipeline (Astronomy I/O)

#### Astronomical Context:
The **Sloan Digital Sky Survey (SDSS)** has mapped millions of galaxies. In optical imaging, galaxies are classified morphologically (e.g., `'Spiral'`, `'Elliptical'`, `'Irregular'`). Distant galaxies undergo cosmological expansion, shifting their light towards redder wavelengths, quantified by the dimensionless **redshift** parameter ($z$).

#### Your Task:
1. Load the SDSS galaxy dataset from `'data/sdss_galaxies.csv'` into DataFrame `df_galaxies`.
2. Filter the galaxies to keep only those that:
   - Have morphological type `galaxy_type == 'Spiral'`
   - **AND** have `redshift > 0.06`
3. Sort the filtered galaxies by `'redshift'` in **ascending order**.
4. Write this filtered DataFrame to a **tab-separated file (TSV)** at `'data/spiral_galaxies_high_z.tsv'` using `sep='\t'` and `index=False`.
5. Read the file back from disk using `pd.read_csv('data/spiral_galaxies_high_z.tsv', sep='\t')` and store it in `df_readback`.


In [ ]:
# --- Exercise 4.1 Work Cell ---
tsv_output_path = 'data/spiral_galaxies_high_z.tsv'

# ==============================================================================
# TODO: Write your code below
# ==============================================================================
df_galaxies = None   # 1. Read 'data/sdss_galaxies.csv'
# 2. Filter: galaxy_type == 'Spiral' and redshift > 0.06
# 3. Sort by redshift ascending
# 4. Save to tsv_output_path with sep='\t', index=False
df_readback = None   # 5. Read back from tsv_output_path
# ==============================================================================

# Inspect results:
if df_readback is not None:
    print("df_readback (from TSV):\n", df_readback)


In [ ]:
# --- AutoEvaluation Cell: Exercise 4.1 ---
try:
    grader.check_4_1(df_galaxies, df_readback, tsv_output_path)
except NameError as e:
    print(f"❌ [NAME ERROR]: {e}")
    print("👉 Ensure you defined: 'df_galaxies' and 'df_readback'.")


---
### 🧬 Exercise 4.2: Gene Expression Filtering & JSON Export (Biology I/O)

#### Biological Context:
Differential expression algorithms (such as DESeq2 or edgeR) produce significance metrics:
- **$\|\log_2(\text{Fold Change})\| \ge 2.0$**: Corresponds to at least a 4-fold increase or decrease in transcription.
- **$\text{Adjusted } p\text{-value} < 10^{-4}$**: Benjamini-Hochberg false discovery rate threshold ensuring statistical significance.

Standard bioinformatics web platforms (such as cBioPortal, Ensembl, or Reactome) consume data formatted as **JSON records** (an array of JSON objects).

#### Your Task:
1. Read the genomic dataset from `'data/gene_expression.json'` into DataFrame `df_genes`.
2. Filter for significantly modulated genes satisfying:
   - Absolute fold change `df_genes['log2_fold_change'].abs() >= 2.0`
   - **AND** `df_genes['adj_p_value'] < 1e-4`
   Store this filtered DataFrame in `df_significant_genes`.
3. Export `df_significant_genes` to a new JSON file at `'data/significant_genes.json'` with:
   - `orient='records'`
   - `indent=2`


In [ ]:
# --- Exercise 4.2 Work Cell ---
json_output_path = 'data/significant_genes.json'

# ==============================================================================
# TODO: Write your code below
# ==============================================================================
df_genes = None              # 1. Read 'data/gene_expression.json'
df_significant_genes = None  # 2. Filter: abs(log2_fold_change) >= 2.0 AND adj_p_value < 1e-4
# 3. Export df_significant_genes to json_output_path (orient='records', indent=2)
# ==============================================================================

# Inspect results:
if df_significant_genes is not None:
    print("df_significant_genes:\n", df_significant_genes[['symbol', 'log2_fold_change', 'adj_p_value', 'pathway']])


In [ ]:
# --- AutoEvaluation Cell: Exercise 4.2 ---
try:
    grader.check_4_2(df_significant_genes, json_output_path)
except NameError as e:
    print(f"❌ [NAME ERROR]: {e}")
    print("👉 Ensure you defined: 'df_significant_genes'.")


---
### 📈 Exercise 4.3: Real-Time FX Stream Parsing via `io.StringIO` (Economics I/O)

#### Financial Context:
Real-time market feeds, webhooks, and REST endpoints frequently return raw text strings rather than files saved on disk. Writing temporary files to disk would incur unnecessary I/O latency. Python's `io.StringIO` allows Pandas to treat an in-memory string buffer as if it were a physical file!

#### Your Task:
Given a raw multi-line string stream representing currency trading data:
```python
raw_exchange_data = """# Daily FX Stream 2026-09-25
Currency,Base,Rate,Volume_Millions
EUR,USD,1.085,420.5
GBP,USD,1.295,310.2
JPY,USD,0.0067,590.8
CHF,USD,1.120,185.0
CAD,USD,0.735,215.4
"""
```

1. Use `io.StringIO(raw_exchange_data)` and `pd.read_csv()` to parse the data into DataFrame `df_fx`.
   *(Hint: Use the argument `comment='#'` to automatically ignore the header comment line).*
2. Add a new column `'Volume_USD_Millions'` to `df_fx` computed as:
   `df_fx['Volume_Millions'] * df_fx['Rate']`.
3. Calculate the total volume traded in USD across all pairs and store the scalar float sum in `total_usd_volume`.


In [ ]:
# --- Exercise 4.3 Work Cell ---
raw_exchange_data = """# Daily FX Stream 2026-09-25
Currency,Base,Rate,Volume_Millions
EUR,USD,1.085,420.5
GBP,USD,1.295,310.2
JPY,USD,0.0067,590.8
CHF,USD,1.120,185.0
CAD,USD,0.735,215.4
"""

# ==============================================================================
# TODO: Write your code below
# ==============================================================================
df_fx = None              # 1. Parse using io.StringIO and comment='#'
# 2. Add 'Volume_USD_Millions'
total_usd_volume = None   # 3. Sum of 'Volume_USD_Millions'
# ==============================================================================

# Inspect results:
if df_fx is not None:
    print("df_fx:\n", df_fx)
print("total_usd_volume (Million USD):", total_usd_volume)


In [ ]:
# --- AutoEvaluation Cell: Exercise 4.3 ---
try:
    grader.check_4_3(df_fx, total_usd_volume)
except NameError as e:
    print(f"❌ [NAME ERROR]: {e}")
    print("👉 Ensure you defined: 'df_fx' and 'total_usd_volume'.")


<a id="5-capstone"></a>
# 🏆 Part 5: Capstone Multi-Domain Challenge

---

### 📖 The Scenario: International Science Consortium Economic Stability Audit
You are appointed as the Data Lead for an international scientific consortium planning multi-billion-dollar observational and genomic initiatives across Eurasia. 

Before committing funding, your team needs an automated pipeline to identify **macroeconomically stable nations** in Europe and Asia that possess high per-capita resources and low financial vulnerability.

### 📋 Capstone Requirements:
1. **Load Data**: Read `'data/economic_indicators.csv'` into DataFrame `df_econ_all`.
2. **Filter Continents**: Subset `df_econ_all` to only include countries located in `'Europe'` or `'Asia'` (Tip: use `.isin(['Europe', 'Asia'])`). Make sure to make a copy (`.copy()`).
3. **Compute Derived Metrics**:
   - `'GDP_per_Capita_USD'`:
     $$\text{GDP\_per\_Capita\_USD} = \frac{\text{gdp\_nominal\_billion\_usd} \times 10^9}{\text{population\_million} \times 10^6}$$
   - `'Economic_Vulnerability_Score'`:
     $$\text{Score} = \text{inflation\_rate\_pct} + \text{unemployment\_rate\_pct} + \frac{\text{debt\_to\_gdp\_pct}}{10.0}$$
4. **Isolate Stable Nations**: Filter for countries with an `'Economic_Vulnerability_Score'` strictly less than `20.0`.
5. **Format Final Table**:
   - Select only the columns: `['country', 'continent', 'GDP_per_Capita_USD', 'Economic_Vulnerability_Score']`.
   - Sort by `'GDP_per_Capita_USD'` in **descending order**.
   - Reset the row index (`reset_index(drop=True)`).
   - Store this clean result in `df_capstone`.
6. **Export**: Save `df_capstone` to `'data/capstone_stable_economies.json'` with `orient='records'` and `indent=2`.


In [ ]:
# --- Capstone Exercise Work Cell ---
capstone_output_path = 'data/capstone_stable_economies.json'

# ==============================================================================
# TODO: Write your code below
# ==============================================================================
df_capstone = None

# Step 1: Load 'data/economic_indicators.csv'
# Step 2: Filter for continent in ['Europe', 'Asia']
# Step 3: Compute 'GDP_per_Capita_USD' and 'Economic_Vulnerability_Score'
# Step 4: Filter Economic_Vulnerability_Score < 20.0
# Step 5: Keep target columns, sort descending by GDP_per_Capita_USD, reset index
# Step 6: Export df_capstone to capstone_output_path (orient='records', indent=2)
# ==============================================================================

# Inspect results:
if df_capstone is not None:
    print("--- 🏛️ Capstone: Stable Partner Nations ---")
    print(df_capstone)


In [ ]:
# --- AutoEvaluation Cell: Capstone Challenge ---
try:
    grader.check_5(df_capstone, capstone_output_path)
except NameError as e:
    print(f"❌ [NAME ERROR]: {e}")
    print("👉 Ensure you defined: 'df_capstone'.")


---
## 🏁 Final Evaluation & Scorecard

Run the cell below at any time to generate your comprehensive practice report card!


In [ ]:
# ==============================================================================
# 📊 FINAL EVALUATION SCORECARD
# ==============================================================================
grader.summary()
